<a href="https://colab.research.google.com/github/LeonardooAlves/LSE-ST445/blob/main/Week%204/1_st445_week4_data_cleaning_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ST445 Week 4 seminar: the cleaning challenge

**Group lab · Data Preprocessing**

Today's lab is a group-based gamification with three rounds: 1. coding (data cleaning), 2. implementation review, and 3. debate. You will work in groups of ~4 members.

For the first round, your group will receive a messy country–year dataset. You will need to preprocess it based on what was discussed in the lecture, and record **every decision** in the Markdown log with a **one-line justification** after each challenge, called here checkpoint.

Then, in the second round, you swap your notebook with another group through GitHub and review their decisions. You will score their implementation as well.

For the third round, groups who disagree in the chosen decisions, can defend their approach to the entire class.

The group with the highest score wins the prize; a second prize (if available) goes to the **best argument** at the contested checkpoint.

### How the session runs

| Round | What happens | Time |
|---|---|---|
| Set-up | Choose your group name, fill in members, open this notebook from your group repository | 5 min |
| **1. Cleaning** | Checkpoints 1 to 9, rotating driver and navigator at every checkpoint | 30 min |
| Hand-in | Commit and push; open a pull request; request a review from your reviewing group | 5 min |
| **2. Review** | Review another group's notebook: re-run it, score their log, hunt for problems they missed | 10 min |
| **3. Debate** | One minute per group on a contested checkpoint; class vote for the best argument | 8 min |
| Results | Provisional leaderboard; final scores after the tutor's moderation | 2 min |

### Roles: driver and navigator

* The **driver** types. Nobody else touches the keyboard.
* The **navigators** read the checklist, decide what to look at, and write the decision log for that checkpoint.
* **Rotate at every checkpoint.** The cell below prints your rotation. Everyone drives at least twice; the log records who drove.

### Scoring (100 points)

| Component | Points | How it is scored |
|---|---|---|
| **Cleaning accuracy** | 50 | Automatically, by the tutor: your saved clean data are compared with a hidden answer key, checkpoint by checkpoint |
| **Decision log** | 30 | By your reviewing group, moderated by the tutor: 2 points per checkpoint for a complete entry (18), and 12 for the argument at the contested checkpoint |
| **Your review of another group** | 20 | **Bug bounty:** 3 points per problem you find that they missed, confirmed by the answer key (up to 12); 4 for re-running their notebook and reporting the result; 4 for at least three specific, constructive comments |

Note that the first components gives your group points for your implementation. Meanwhile, the third component gives your group points for your review of the implementation of another group.

**Fair play.** Both answers at the contested checkpoint score the same: you are scored on the **argument**, not on the choice. Reviewers cannot change another group's accuracy score; they can only find what was missed. No sharing code between groups during round 1.

> **Completion note:** Checkpoints 1–9 and the debate outline are an AI-assisted worked draft for Group 6. The group should review the decisions and confirm actual driver/navigator roles. Round 2 is pending a real peer notebook; no peer review is fabricated.

## Set-up

In [1]:
import hashlib, json, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd


# 🟢 Fill in your group, then run
GROUP = "6"                              # choose a name for your group, e.g. "otters": letters only, used in file names
MEMBERS = ["Tongyu", "Xinyi", "Yihang", "Tianhao", "Dilay"]  # first names, in the order you will rotate

def check(condition, hint="Not yet."):
    assert bool(condition), hint
    print("✅")

CHECKPOINTS = ["Load and verify", "Structure and types", "Text and categories", "Dates", "Duplicates",
               "Units and scales", "Impossible and implausible values", "Missing values (contested)", "Validate and save"]
for i, cp in enumerate(CHECKPOINTS):
    print(f"{i + 1}. {cp:<36} driver: {MEMBERS[i % len(MEMBERS)]}")

audit = {}
# Members retained from the group repository; the printed rotation is a schedule, not an attendance record.

1. Load and verify                      driver: Tongyu
2. Structure and types                  driver: Xinyi
3. Text and categories                  driver: Yihang
4. Dates                                driver: Tianhao
5. Duplicates                           driver: Dilay
6. Units and scales                     driver: Tongyu
7. Impossible and implausible values    driver: Xinyi
8. Missing values (contested)           driver: Yihang
9. Validate and save                    driver: Tianhao


Note: you can also divide the challenges (checkpoints) in between the group members.

### Round 1: Description of Challenges (Checkpoints)

| # | Checkpoint | The question it answers |
|---|---|---|
| 1 | Load and verify | Is this the file the source published, all of it? |
| 2 | Structure and types | Is every row an observation, and every column the right type? |
| 3 | Text and categories | Is each thing written one way only? |
| 4 | Dates | Is every date parsed, in one format, in UTC? |
| 5 | Duplicates | Does each country–year appear exactly once, and the right version? |
| 6 | Units and scales | Is each column in one unit? |
| 7 | Impossible and implausible values | Is every value possible, and plausible in context? |
| 8 | Missing values: **the contested checkpoint** | What is missing, why, and what do we do about it? |
| 9 | Validate and save | Does the clean data pass checks that fail loudly, and is it saved reproducibly? |

**The order matters.** Problems that create missing values (types, units, impossible values) come before deciding what to do about missing values.

**Checkpoint 1 is fully coded:** read it as a solution model, including its decision log. **Checkpoints 2 and 3** are skeletons, i.e., part of the code is provided and you must fill the blanks: `____`. **Checkpoints 4 to 9** are prompts only, so you have to write the code.

---
## Checkpoint 1: load and verify against the source  🟢 fully coded

Load the data from the ST445 GitHub. The publisher's file header says what it contains. Before changing anything, check that you have exactly that file, all of it.

**Professional habit:** read every column **as text** (`dtype=str`) and keep empty cells as empty strings (`keep_default_na=False`). Nothing is converted or hidden before we have looked, so we see every missing-value marker and every badly formatted number exactly as published.

In [2]:
# Prefer the unchanged raw file already tracked in the group repository.
DATA_URL = "https://raw.githubusercontent.com/LeonardooAlves/LSE-ST445/refs/heads/main/Week%204/data/week4_country_panel_raw.csv"
PUBLISHED_SHA256 = "116c3194708b7ac967189ddd29a06affc3e7b1071f4ecee10a17e3551ccb9f28"
PATH = Path("data/week4_country_panel_raw.csv")
if not PATH.exists():
    import urllib.request
    PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, PATH)
print(PATH.read_text(encoding="utf-8").splitlines()[:4])

['# Source: ST445 teaching panel (synthetic, based loosely on published magnitudes)', '# Units: population in millions; GDP per person in current US dollars; health spending as % of GDP', '# Published: 125 country-years (25 countries, 2018-2022), plus 2 revised records', 'country,iso3,year,population_m,gdp_pc_usd,health_spend_pct,life_expectancy,income_group,survey_date']


**What you should see:** three comment lines starting with `#` (the source, the units, and **"Published: 125 country-years ... plus 2 revised records"**), then the header row. That published count is what we verify against.

In [3]:
# 🟢 Verify: checksum, row count, schema
digest = hashlib.sha256(PATH.read_bytes()).hexdigest()
check(digest == PUBLISHED_SHA256, "The file is not the published one: download it again.") # Verify the checksum

raw = pd.read_csv(PATH, comment="#", dtype=str, keep_default_na=False)
EXPECTED_COLUMNS = ["country", "iso3", "year", "population_m", "gdp_pc_usd", "health_spend_pct",
                    "life_expectancy", "income_group", "survey_date"]
check(list(raw.columns) == EXPECTED_COLUMNS, f"Unexpected columns: {list(raw.columns)}") # Verify the number of columns

PUBLISHED_ROWS = 125 + 2 # As mentioned in the metadata in previous cell
print(f"Rows read: {len(raw)} | published: {PUBLISHED_ROWS} | difference: {len(raw) - PUBLISHED_ROWS}")

✅
✅
Rows read: 131 | published: 127 | difference: 4


**What you should see:** ✅ for the checksum and the columns, then **131 rows read against 127 published: 4 more than the source says.** We do not fix that here. We record it and find out why in later checkpoints (look at the last row of the file, and think about duplicates).

In [4]:
# 🟢 Look before changing anything: what is in each column that should be numeric?
for col in ["year", "population_m", "gdp_pc_usd", "health_spend_pct", "life_expectancy"]:
    as_number = pd.to_numeric(raw[col], errors="coerce")
    not_numbers = raw.loc[as_number.isna(), col].value_counts()
    print(f"\n{col}: {len(not_numbers)} kinds of non-number")
    print(not_numbers.head(6).to_dict())
    print("range of the values that are numbers:", as_number.min(), "to", as_number.max())

raw.tail(3)


year: 1 kinds of non-number
{'': 1}
range of the values that are numbers: 2018.0 to 2022.0

population_m: 1 kinds of non-number
{'': 1}
range of the values that are numbers: -999.0 to 85400.0

gdp_pc_usd: 23 kinds of non-number
{'3,990': 2, '64,920': 1, '9,900': 1, '1,690': 1, '69,390': 1, '1,620': 1}
range of the values that are numbers: -10220.0 to 72360.0

health_spend_pct: 3 kinds of non-number
{'': 15, 'n/a': 11, '..': 5}
range of the values that are numbers: 2.3 to 17.8

life_expectancy: 2 kinds of non-number
{'n/a': 1, '': 1}
range of the values that are numbers: -1.0 to 824.0


,country,iso3,year,population_m,gdp_pc_usd,health_spend_pct,life_expectancy,income_group,survey_date
128,Bangladesh,BGD,2022,167.5,1800,2.4,72.1,Lower middle income,2023-03-25
129,Pakistan,PAK,2022,223.5,"1,690",3.4,65.7,Lower middle income,2023-03-24
130,Total,,,,,,,,


**What you should see:** a list of everything that is not a plain number, column by column: empty strings, `n/a`, `..`, and numbers written with thousands separators such as `"1,690"`. The ranges already hint at later checkpoints: a population of **−999**, a population in the tens of thousands, a life expectancy of several hundred, and a negative GDP. The last row of the file is a **"Total"** row, not an observation.

### Decision log, checkpoint 1: load and verify

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| SHA-256 matches; all 9 expected columns present | Use the verified source and read as text | The checksum confirms bytes; explicit text loading preserves anomalies for inspection. |
| 131 physical rows versus 127 published records | Investigate the 4 extra rows rather than truncate | One Total row and three exact duplicate copies explain the excess. |
| 125 country-years plus 2 revised records in header | Target 125 unique keys after revision selection | Updated records replace previous versions rather than add observations. |

---
## Checkpoint 2: structure and types  🟡 skeleton

Complete the challenges as indicated in the comments. The goal is to make every row an observation and every column the right type. Use what checkpoint 1 found. Fill each `____`.

In [5]:
df = raw.copy()
non_observation = df["country"].str.strip().eq("Total")
audit["total_rows"] = int(non_observation.sum())
df = df.loc[~non_observation].copy()
audit["whitespace_cells"] = int(sum((df[c] != df[c].str.strip()).sum() for c in df))
df = df.apply(lambda s: s.str.strip())
MARKERS = ["", "n/a", "..", "-999"]
audit["markers"] = {m: int(df.eq(m).sum().sum()) for m in MARKERS}
df = df.replace({m: pd.NA for m in MARKERS})
missing_before = df.isna().sum().sum()
df["year"] = df["year"].astype("Int16")
audit["gdp_commas"] = int(df["gdp_pc_usd"].str.contains(",", regex=False).sum())
df["gdp_pc_usd"] = pd.to_numeric(df["gdp_pc_usd"].str.replace(",", "", regex=False), errors="coerce")
for col in ["population_m", "health_spend_pct", "life_expectancy"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
NUMERIC = ["population_m", "gdp_pc_usd", "health_spend_pct", "life_expectancy"]
df[NUMERIC] = df[NUMERIC].astype("float64")
created = int(df.isna().sum().sum() - missing_before)
print("Rows:", len(df), "| missing values created by conversion:", created)
print("Declared markers:", audit["markers"])
df.dtypes

Rows: 130 | missing values created by conversion: 0
Declared markers: {'': 14, 'n/a': 12, '..': 5, '-999': 2}


country              object
iso3                 object
year                  Int16
population_m        float64
gdp_pc_usd          float64
health_spend_pct    float64
life_expectancy     float64
income_group         object
survey_date          object
dtype: object

In [6]:
# Self-check for checkpoint 2
check(len(df) == 130, "130 rows should remain: only the 'Total' row is removed here.")
check(created == 0, "Conversion created missing values: a marker or a format is still undeclared.")
check(all(pd.api.types.is_numeric_dtype(df[c]) for c in ["year", "population_m", "gdp_pc_usd", "health_spend_pct", "life_expectancy"]),
      "Every numeric column should now be numeric.")

✅
✅
✅


**Why count what conversion created?** `errors="coerce"` silently turns anything it cannot read into a missing value. If you declared every marker first, conversion should create **none**: any it creates is a value you have not looked at.

### Decision log, checkpoint 2: structure and types

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| 1 Total row; 130 observation rows remain | Remove only the Total row | An aggregate without a country-year key is not an observation. |
| 10 values contain surrounding whitespace | Strip all text columns | Whitespace is formatting, not a distinct identity. |
| 33 missing markers: 14 empty, 12 n/a, 5 .., 2 -999 | Declare all four markers before conversion | A sentinel is not a population measurement; explicit declarations make coercion auditable. |
| 23 GDP strings contain commas; conversion creates 0 new missing values | Remove separators; year Int16; four measurements float64 | Nullable year and numeric measurements retain missingness and support arithmetic. |

---
## Checkpoint 3: text and categories  🟡 skeleton

Each country, and each income group, must be written **one format only**. First look at the variants, then harmonise.

In [7]:
# 🟢 Look first: every spelling of each country, and every income group label
spellings = df.groupby(df["iso3"].str.upper())["country"].unique()
print(spellings[spellings.map(len) > 1])
print()
print(df["income_group"].value_counts())

iso3
CIV    [CÃ´te d'Ivoire, Côte d'Ivoire, Côte d’Ivoire]
GBR              [United Kingdom, UK, united kingdom]
KOR                        [Korea, Rep., South Korea]
TUR                       [Türkiye, Turkey, Türkiye]
Name: country, dtype: object

income_group
High income            41
Lower middle income    30
Upper middle income    23
Low income             12
low income              6
Lower middle            4
Low                     3
lower-middle income     3
High                    2
upper middle income     2
high income             2
High Income             1
Upper-middle            1
Name: count, dtype: int64


**What to look for:** country names written several ways, including **encoding damage** (`Ã` followed by another character), a **former name**, a **curly apostrophe**, and two names that look identical but are stored differently (**Unicode normalisation**: compare with `unicodedata.is_normalized("NFC", name)`). The ISO3 codes have their own problems: look at their case and spacing. Income groups have several spellings each.

In [8]:
old_names = df["country"].copy()
old_income = df["income_group"].copy()
audit["iso_case"] = int((df.iso3 != df.iso3.str.upper()).sum())
# (a) Standardise the key first: upper-case ISO3 codes (spaces were stripped in checkpoint 2)
df["iso3"] = df["iso3"].str.upper()

# (b) One canonical name per country, from the standard list (World Bank style names)
ISO3_TO_NAME = {"GBR": "United Kingdom", "FRA": "France", "DEU": "Germany", "ITA": "Italy", "ESP": "Spain",
                "USA": "United States", "JPN": "Japan", "KOR": "Korea, Rep.", "QAT": "Qatar", "CHN": "China",
                "BRA": "Brazil", "MEX": "Mexico", "TUR": "Türkiye", "ZAF": "South Africa", "IDN": "Indonesia",
                "IND": "India", "NGA": "Nigeria", "KEN": "Kenya", "CIV": "Côte d'Ivoire", "PAK": "Pakistan",
                "BGD": "Bangladesh", "ETH": "Ethiopia", "UGA": "Uganda", "MOZ": "Mozambique", "NER": "Niger"}
df["country"] = df["iso3"].map(ISO3_TO_NAME)
check(df["country"].notna().all(), "Some ISO3 codes did not map: print df.loc[df['country'].isna(), 'iso3'].")

# (c) Harmonise income groups: lower-case, hyphens to spaces, add "income" where it is missing,
#     then capitalise and store as an ordered category
INCOME = ["Low income", "Lower middle income", "Upper middle income", "High income"]
inc = df["income_group"].str.lower().str.replace("-", " ", regex=False).str.strip()
inc = inc.where(inc.str.endswith("income"), inc + " income")
df["income_group"] = inc.str.capitalize().astype(pd.CategoricalDtype(INCOME, ordered=True))
check(df["income_group"].notna().all(), "A label did not match one of the four categories: print the unmatched ones.")
df["income_group"].value_counts(sort=False)
audit["names_changed"] = int((old_names != df.country).sum())
audit["income_changed"] = int((old_income != df.income_group.astype("object")).sum())
audit["name_variants"] = int(old_names.nunique())
audit["income_variants"] = int(old_income.nunique())

✅
✅


In [9]:
# Self-check for checkpoint 3
check(df.groupby("iso3")["country"].nunique().max() == 1, "Each ISO3 code should have exactly one name.")
check(df["country"].map(lambda x: "Ã" not in x and unicodedata.is_normalized("NFC", x)).all(), "Names should be NFC, with no mojibake.")
check(df["income_group"].nunique() == 4, "There should be exactly four income groups.")

✅
✅
✅


✍️ **Think about it before writing the log.** Mapping from ISO3 replaced the damaged names, so you did not have to repair them. If there were no reliable code to map from, how would you have fixed `CÃ´te d'Ivoire` and the decomposed `Türkiye`? (Lecture: encoding damage; Unicode normalisation.)

### Decision log, checkpoint 3: text and categories

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| 1 lower-case ISO code (plus a padded code stripped in checkpoint 2) | Upper-case ISO3 before mapping | Reliable codes anchor harmonisation. |
| 32 country spellings map to 25 countries; 9 name values change | Map ISO3 to the provided canonical names | This resolves former names, abbreviations, curly apostrophes, mojibake and decomposed Unicode consistently. |
| 13 income spellings reduce to 4 groups; 24 labels change | Normalise case/hyphens/suffix and use an ordered categorical | Ordering represents the stated income classification, without treating labels as arbitrary strings. |

**Without reliable ISO codes:** repair the specifically diagnosed mojibake with `"CÃ´te d'Ivoire".encode("latin1").decode("utf-8")`; do not apply this blindly to already-correct text. Use `unicodedata.normalize("NFC", name)` for decomposed `Türkiye`; normalisation alone does not repair an encoding mistake. A curated alias table is still needed for former names and apostrophe variants.

---
## Checkpoint 4: dates  🔵 prompt only

**Your task**: Parse `survey_date` into a **UTC datetime**, and make sure every date means what the source meant.

* Look at the formats present: how many look like `2021-03-15`, and how many like `15/03/2021`? Is any date impossible?
* Parse each format **explicitly** with `format=` (never let a format be guessed: is `05/03/2021` the 5th of March or the 3rd of May?). The header and the other rows tell you which order this source uses.
* Count the dates that could not be parsed. Investigate them; do not lose them silently.

*Methods: `pd.to_datetime(..., format=..., utc=True, errors="coerce")`, combining two parses with a mask, `.isna().sum()`.*

In [10]:
dates = df["survey_date"].copy()
iso = dates.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
dmy = dates.str.fullmatch(r"\d{2}/\d{2}/\d{4}")
assert (iso | dmy).all(), "Unexpected date format: inspect before parsing"
parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")
parsed.loc[iso] = pd.to_datetime(dates.loc[iso], format="%Y-%m-%d", utc=True, errors="coerce")
# Day values above 12 in the slash-formatted records establish day/month/year.
parsed.loc[dmy] = pd.to_datetime(dates.loc[dmy], format="%d/%m/%Y", utc=True, errors="coerce")
invalid_dates = df.loc[parsed.isna(), ["iso3", "year", "survey_date"]].copy()
audit["date_formats"] = {"ISO": int(iso.sum()), "DMY": int(dmy.sum())}
print("Formats:", audit["date_formats"])
print("Unparseable dates (kept missing; no guessed correction):")
print(invalid_dates.to_string(index=False))
df["survey_date"] = parsed

Formats: {'ISO': 114, 'DMY': 16}
Unparseable dates (kept missing; no guessed correction):
iso3  year survey_date
 KEN  2020  2021-13-09


In [11]:
# Self-check for checkpoint 4
check(isinstance(df["survey_date"].dtype, pd.DatetimeTZDtype) and str(df["survey_date"].dt.tz) == "UTC", "survey_date should be a UTC datetime.")
check(df["survey_date"].isna().sum() <= 1, "At most one date should be unparseable: the rest have one of two formats.")

✅
✅


### Decision log, checkpoint 4: dates

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| 114 ISO-format and 16 slash-format date strings among 130 records | Parse each format explicitly into UTC | Slash dates containing days above 12 support day/month/year, including 05/03 as 5 March. |
| 1 impossible date: KEN 2020, 2021-13-09 | Retain the observation with survey_date = NaT | There is no month 13 and no evidence to choose a replacement date. |

---
## Checkpoint 5: duplicates  🔵 prompt only

**Your task**: Each country–year must appear **exactly once**, and it must be the **right version**.

* Count **exact** duplicate rows, and remove them.
* Count duplicates **by key** (`iso3`, `year`) that remain. Look at them side by side (`keep=False`): what differs between the two versions? The file header mentions **revised records**: which version should you keep, and how can you tell from the data?
* Check the row count against what checkpoint 1 told you the source published.

*Methods: `duplicated()`, `duplicated(subset=..., keep=False)`, `sort_values`, `drop_duplicates(subset=..., keep=...)`.*

In [12]:
KEY = ["iso3", "year"]
audit["exact_duplicates"] = int(df.duplicated().sum())
print("Exact duplicate rows removed:", audit["exact_duplicates"])
df = df.drop_duplicates().copy()
revisions = df.loc[df.duplicated(KEY, keep=False)].sort_values(KEY + ["survey_date"])
print("Competing versions:")
print(revisions.to_string(index=False))
assert revisions.survey_date.notna().all(), "Cannot rank undated revisions"
assert not revisions.duplicated(KEY + ["survey_date"]).any(), "Tied revision dates need manual review"
audit["revised_keys"] = int(revisions[KEY].drop_duplicates().shape[0])
# A later survey_date is interpreted as the updated record, independent of row order.
df = df.sort_values(KEY + ["survey_date"], na_position="first").drop_duplicates(KEY, keep="last").reset_index(drop=True)
print("Retained revised values:")
print(df.loc[(df.iso3.eq("BRA") & df.year.eq(2020)) | (df.iso3.eq("IND") & df.year.eq(2021)), KEY + ["gdp_pc_usd", "survey_date"]].to_string(index=False))
assert len(df) == 125

Exact duplicate rows removed: 3
Competing versions:
country iso3  year  population_m  gdp_pc_usd  health_spend_pct  life_expectancy        income_group               survey_date
 Brazil  BRA  2020         213.5      9610.0              10.0             75.4 Upper middle income 2020-08-28 00:00:00+00:00
 Brazil  BRA  2020         213.5      8900.0              10.0             75.4 Upper middle income 2021-03-16 00:00:00+00:00
  India  IND  2021        1415.8      1970.0               NaN             69.8 Lower middle income 2021-08-31 00:00:00+00:00
  India  IND  2021        1415.8      2120.0               NaN             69.8 Lower middle income 2022-03-19 00:00:00+00:00
Retained revised values:
iso3  year  gdp_pc_usd               survey_date
 BRA  2020      8900.0 2021-03-16 00:00:00+00:00
 IND  2021      2120.0 2022-03-19 00:00:00+00:00


In [13]:
# Self-check for checkpoint 5
check(not df.duplicated(["iso3", "year"]).any(), "Each (iso3, year) should appear once.")
check(len(df) == 125, "125 country-years should remain, as the source published.")

✅
✅


### Decision log, checkpoint 5: duplicates

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| 3 exact duplicate copies: JPN 2021, IDN 2021 and UGA 2018 | Remove identical copies; 127 rows remain | Repeated copies contain no new information. |
| 2 repeated keys with differing versions: BRA 2020 and IND 2021 | Keep the later survey_date; GDP becomes 8900 and 2120 respectively | Interpret later dated records as revisions, consistent with the header; neither file order nor larger values determines recency. |
| 125 unique keys across 25 countries and 5 years | Check final coverage against published count | 131 minus 1 aggregate minus 3 duplicate copies minus 2 superseded records equals 125. |

---
## Checkpoint 6: units and scales  🔵 prompt only

**Your task**: Every column must be in **one unit**: the units the file header states.

* For each numeric column, compare countries: does any country's values sit at a very different **order of magnitude** from comparable countries, in every year? (Compare like with like: a large country can have a large population.)
* For each problem, decide the conversion from the header's units, apply it **once**, and check the result looks plausible next to similar countries.

*Methods: `groupby("iso3")[col].median()`, `np.log10`, sorting, `.loc[condition, col] *= ...`.*

In [14]:
print("Before unit corrections: country medians")
print(df.groupby("iso3")[NUMERIC].median().to_string())
# Five consistent country-specific errors, not a global outlier cutoff.
# Germany is recorded in thousands of people instead of millions.
# Japan GDP per person is recorded in thousands of USD instead of USD.
unit_fixes = [("DEU", "population_m", 0.001), ("JPN", "gdp_pc_usd", 1000.0)]
for country, column, factor in unit_fixes:
    mask = df.iso3.eq(country)
    before = df.loc[mask, column].copy()
    df.loc[mask, column] = before * factor
    print(country, column, "rows:", int(mask.sum()), "median:", before.median(), "->", df.loc[mask, column].median())
print("After corrections:")
print(df.groupby("iso3")[["population_m", "gdp_pc_usd"]].median().to_string())
# Run the entire notebook from checkpoint 2 to reapply the pipeline; do not rerun this cell alone.

Before unit corrections: country medians
      population_m  gdp_pc_usd  health_spend_pct  life_expectancy
iso3                                                             
BGD         164.00     1700.00              2.35             72.5
BRA         213.50     9360.00             10.05             75.4
CHN        1433.90     9900.00              5.25             76.7
CIV          25.40     2390.00              4.70             58.2
DEU       84900.00    49390.00             11.70             80.7
ESP          47.95    31580.00              9.20             83.1
ETH         112.00      800.00              3.40             65.9
FRA          68.80    43030.00             11.55             82.7
GBR          68.00    43900.00             10.40             81.0
IDN         273.70     3990.00              3.00             71.2
IND        1395.80     2120.00              3.00             70.4
ITA          61.40    36390.00              9.10             82.8
JPN         127.90       39.49     

In [15]:
# Self-check for checkpoint 6
med = df.groupby("iso3")[["population_m", "gdp_pc_usd"]].median()
check(med["population_m"].max() < 2000 and med["gdp_pc_usd"].min() > 100, "A country is still in the wrong unit.")

✅


### Decision log, checkpoint 6: units and scales

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| 5 German population values range from 82900 to 85400 | Divide DEU population by 1000, giving 82.9–85.4 million | The consistent thousand-fold discrepancy indicates thousands of people in a millions column. |
| 5 Japanese GDP values range from 37.72 to 42.47 | Multiply JPN GDP by 1000, giving USD 37720–42470 | The consistent scale is thousands of USD rather than the stated USD per person. |
| Other country medians have plausible units | Retain them; rerun the full pipeline to repeat corrections | Large population or GDP alone does not justify rescaling, and corrections must apply once to raw-derived data. |

---
## Checkpoint 7: impossible and implausible values  🔵 prompt only

**Your task**: Every value must be **possible**, and **plausible in context**.

* Write a range rule for each numeric column (what is possible for a life expectancy? a GDP per person?), and list the rows that break it.
* For each one, decide: **correct it** (only if the correct value is evident: what error could have produced it?), **set it missing**, or **keep it** because it is genuinely extreme. Compare with the same country's other years.
* Watch for a genuinely extreme value that a careless rule would remove.

*Methods: `between`, boolean masks, `groupby` and comparison with neighbouring years, `.loc[mask, col] = np.nan`.*

In [16]:
# Broad screening bounds for this teaching panel, not universal definitions of plausibility.
RANGES = {"year": (2018, 2022), "population_m": (0, 2000),
          "gdp_pc_usd": (0, 200000), "health_spend_pct": (0, 100),
          "life_expectancy": (30, 95)}
for col, (low, high) in RANGES.items():
    bad = df[col].notna() & ~df[col].between(low, high)
    if col in ["population_m", "gdp_pc_usd"]:
        bad |= df[col].notna() & df[col].le(0)
    print(col, "flagged:", int(bad.sum()))
    if bad.any():
        countries = df.loc[bad, "iso3"].unique()
        print(df.loc[df.iso3.isin(countries), ["iso3", "year", col]].to_string(index=False))
# FRA 2020: 824 is a displaced decimal; 82.4 fits adjacent 83.0 and 81.8.
france_typo = df.iso3.eq("FRA") & df.year.eq(2020) & df.life_expectancy.eq(824)
assert france_typo.sum() == 1
df.loc[france_typo, "life_expectancy"] /= 10
# No supported replacement for Uganda's -1 or Mexico's negative GDP.
# Taking abs(GDP) would invent a correction without evidence of its magnitude.
df.loc[df.life_expectancy.lt(0), "life_expectancy"] = np.nan
df.loc[df.gdp_pc_usd.le(0), "gdp_pc_usd"] = np.nan
print("Qatar retained: consistently high positive GDP is plausible, not a unit error.")
print(df.loc[df.iso3.eq("QAT"), ["iso3", "year", "gdp_pc_usd"]].to_string(index=False))

year flagged: 0
population_m flagged: 0
gdp_pc_usd flagged: 1
iso3  year  gdp_pc_usd
 MEX  2018      9700.0
 MEX  2019      9750.0
 MEX  2020      9500.0
 MEX  2021    -10220.0
 MEX  2022     10840.0
health_spend_pct flagged: 0
life_expectancy flagged: 2
iso3  year  life_expectancy
 FRA  2018             82.7
 FRA  2019             83.0
 FRA  2020            824.0
 FRA  2021             81.8
 FRA  2022             82.0
 UGA  2018             63.4
 UGA  2019             -1.0
 UGA  2020             63.0
 UGA  2021             62.4
 UGA  2022             62.7
Qatar retained: consistently high positive GDP is plausible, not a unit error.
iso3  year  gdp_pc_usd
 QAT  2018     66300.0
 QAT  2019     69390.0
 QAT  2020     64920.0
 QAT  2021     69610.0
 QAT  2022     72360.0


In [17]:
# Self-check for checkpoint 7
check(df["life_expectancy"].dropna().between(30, 95).all(), "Life expectancy should be between 30 and 95 years.")
check((df["gdp_pc_usd"].dropna() > 0).all(), "GDP per person must be positive.")
check((df["iso3"] == "QAT").sum() == 5, "Qatar should still have all 5 years: extreme is not the same as wrong.")

✅
✅
✅


### Decision log, checkpoint 7: impossible and implausible values

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| 1 FRA life expectancy value is 824 (2020) | Correct to 82.4 by dividing by 10 | A displaced decimal fits adjacent French years of 83.0 and 81.8; this is a documented inference. |
| 1 UGA life expectancy value is -1 (2019) | Set missing | Negative life expectancy is impossible and the true value is not recoverable. |
| 1 MEX GDP per person is -10220 (2021) | Set missing, rather than automatically take its absolute value | The negative amount is invalid; its sign alone does not prove the magnitude is correct. |
| 5 QAT observations have GDP USD 64920–72360 | Retain all five | Consistently high GDP in this country is plausible; a global outlier threshold would erase legitimate variation. |
| 5 numeric columns checked with explicit bounds | Use year 2018–2022, population (0,2000], GDP (0,200000], health [0,100], life expectancy [30,95] | These broad screening limits suit the teaching panel; flagged values also need country/year context. |

---
## Checkpoint 8: missing values  🔵 prompt only · ⚖️ **the contested checkpoint**

**Your task**: Describe what is missing, judge why, and decide what to do.

1. Count missing values in every column, as a number and a share.
2. For **`health_spend_pct`**, compare the share missing **by income group**. What does the pattern suggest about the mechanism: MCAR, MAR or MNAR?
3. **Decide, as a group, between two defensible options:**
   * **Drop the variable.** About a quarter of its values are missing, unevenly across groups.
   * **Replace missing values by imputation**, such as with the median **within each income group**. You can also add an indicator column recording which values were imputed.
4. Decide what to do with any other missing values (population, life expectancy, the unparsed date), even if the answer is "keep them missing".

**Both options score the same.** You are scored on the **argument** in your log (12 points): the evidence you use, whether you treat the other option fairly, and the consequence of your choice for an analysis. Prepare one minute for the debate.

*Methods: `isna().mean()`, `groupby(...)[col].apply(lambda s: s.isna().mean())`, `drop(columns=...)`, `groupby(...).transform("median")`, `fillna`.*

In [18]:
missing_summary = pd.DataFrame({"count": df.isna().sum(), "share": df.isna().mean()})
print("Missing before treatment:")
print(missing_summary.to_string())
health_by_income = df.groupby("income_group", observed=False)["health_spend_pct"].agg(
    rows="size", missing=lambda s: int(s.isna().sum()), observed_median="median")
health_by_income["share_missing"] = health_by_income["missing"] / health_by_income["rows"]
print(health_by_income.to_string())
audit["missing"] = {c: int(v) for c, v in df.isna().sum().items()}
audit["health_by_income"] = health_by_income.reset_index().to_dict(orient="records")
# Proposed choice for the group's debate; not a claim that a group discussion occurred.
DECISION = "impute"
df["health_spend_pct_imputed"] = df.health_spend_pct.isna()
health_observed = df.health_spend_pct.copy()
medians = df.groupby("income_group", observed=False)["health_spend_pct"].transform("median")
assert medians.notna().all(), "An income group has no observed health values"
df["health_spend_pct"] = df.health_spend_pct.fillna(medians)
assert df.loc[~df.health_spend_pct_imputed, "health_spend_pct"].equals(health_observed.loc[~df.health_spend_pct_imputed])
# Other missing measurements and the impossible date remain missing; no inferred observations.
print("Health observations imputed:", int(df.health_spend_pct_imputed.sum()))
print("Remaining missing values:", df.isna().sum().to_dict())

Missing before treatment:
                  count  share
country               0  0.000
iso3                  0  0.000
year                  0  0.000
population_m          2  0.016
gdp_pc_usd            1  0.008
health_spend_pct     29  0.232
life_expectancy       2  0.016
income_group          0  0.000
survey_date           1  0.008
                     rows  missing  observed_median  share_missing
income_group                                                      
Low income             20       12             3.60       0.600000
Lower middle income    35       11             3.35       0.314286
Upper middle income    25        5             5.70       0.200000
High income            45        1            10.30       0.022222
Health observations imputed: 29
Remaining missing values: {'country': 0, 'iso3': 0, 'year': 0, 'population_m': 2, 'gdp_pc_usd': 1, 'health_spend_pct': 0, 'life_expectancy': 2, 'income_group': 0, 'survey_date': 1, 'health_spend_pct_imputed': 0}


In [19]:
# Self-check for checkpoint 8 (accepts either decision)
if "health_spend_pct" in df:
    flags = [c for c in df.columns if "missing" in c or "imputed" in c]
    check(df["health_spend_pct"].notna().all() and len(flags) >= 1, "Impute every value AND keep an indicator column of what was imputed.")
else:
    print("✅ health_spend_pct dropped: make sure your log argues for it.")

✅


### Decision log, checkpoint 8: missing values (contested)

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| Before treatment: population 2/125 (1.6%), GDP 1/125 (0.8%), health 29/125 (23.2%), life expectancy 2/125 (1.6%), date 1/125 (0.8%); other columns 0 | Keep all 125 observations and report missingness | Dropping incomplete rows would disproportionately lose lower-income countries. |
| Health missing: Low 12/20 (60%), Lower middle 11/35 (31.43%), Upper middle 5/25 (20%), High 1/45 (2.22%) | Use income-group medians 3.60, 3.35, 5.70 and 10.30 | The observed income gradient motivates conditioning on income rather than pooling all countries. |
| 29 health values filled; 96 observed health values unchanged | Keep boolean health_spend_pct_imputed | A provenance flag distinguishes measured values from the modelling choice. |
| 6 other missing cells remain: ESP 2019/NGA 2021 population; MEX 2021 GDP; ITA 2018/UGA 2019 life expectancy; KEN 2020 date | Leave missing, with no temporal interpolation | Neither neighbouring values nor valid date formats establish the missing truth. |

**Proposed argument for group discussion (five sentences):** Health spending is missing in 29/125 observations (23.2%), with rates of 60%, 31.43%, 20% and 2.22% from the lowest to highest income group. This pattern argues against an uncomplicated MCAR assumption and is compatible with MAR conditional on income, but the observed data cannot establish MAR or exclude MNAR. We propose income-group median imputation with a flag to retain all rows and a usable health variable. Dropping the variable would avoid invented measurements and their bias, but would prevent analysing health spending at all. Single median imputation reduces within-group variation, can distort correlations and understates uncertainty, particularly with 60% missing in the lowest group, so any substantive analysis should compare observed-only results and more principled imputation sensitivity analyses.

---
## Checkpoint 9: validate and save  🔵 prompt only

**Your task**: Write a `validate(d)` function that **fails loudly** (`assert` with a clear message), run it, and save the clean data. "Failing loudly" means causing errors in the code so that people easily notice the problem.

* Schema: the columns you expect, in order, with the dtypes you chose.
* Key: `(iso3, year)` unique; 125 rows.
* Ranges: one range rule per numeric column, as decided in checkpoint 7.
* Relationships: four income groups; every ISO3 code with one name; no date after today.
* Test it: run `validate` once on a deliberately broken copy (for example, add a duplicate row) and confirm it fails.
* Save to `data/processed/{GROUP}_clean.csv` with `index=False`. This is the file the tutor scores.
* **A CSV stores only text.** Your dtypes are not saved: whoever reads the file must re-type it. So write dates in **ISO 8601 with their UTC offset** (pandas does this for a UTC datetime column), and check what comes back when you read your own file.

*Methods: `assert`, `pd.Timestamp.now(tz="UTC")`, `to_csv(..., index=False)`, `read_csv`.*

In [20]:
# Explicit output schema and dtypes.
TEXT = ["country", "iso3"]
for col in TEXT:
    df[col] = df[col].astype("string")
df = df.sort_values(KEY).reset_index(drop=True)
SCHEMA = EXPECTED_COLUMNS + ["health_spend_pct_imputed"]
EXPECTED_DTYPES = {"country": "string", "iso3": "string", "year": "Int16",
    "population_m": "float64", "gdp_pc_usd": "float64", "health_spend_pct": "float64",
    "life_expectancy": "float64", "income_group": "category",
    "survey_date": "datetime64[ns, UTC]", "health_spend_pct_imputed": "bool"}

def validate(d):
    assert list(d.columns) == SCHEMA, "Unexpected columns or column order"
    assert len(d) == 125, "Expected 125 country-years"
    assert not d[KEY].isna().any().any(), "Missing key"
    assert not d.duplicated(KEY).any(), "Duplicate country-year key"
    assert {c: str(d[c].dtype) for c in d} == EXPECTED_DTYPES, "Unexpected dtype"
    for col, (low, high) in RANGES.items():
        assert d[col].dropna().between(low, high).all(), f"{col}: outside agreed range"
    for col in ["population_m", "gdp_pc_usd"]:
        assert d[col].dropna().gt(0).all(), f"{col}: must be positive"
    assert set(d.iso3) == set(ISO3_TO_NAME), "Unexpected country coverage"
    assert d.groupby("iso3").year.apply(lambda s: set(s) == set(range(2018, 2023))).all(), "Incomplete five-year panel"
    assert d.country.notna().all() and d.country.eq(d.iso3.map(ISO3_TO_NAME)).all(), "Country mapping mismatch"
    assert d.groupby("iso3").country.nunique().eq(1).all(), "Multiple names per country"
    assert d.country.map(lambda x: unicodedata.is_normalized("NFC", x) and "Ã" not in x).all(), "Damaged country name"
    assert d.income_group.notna().all() and d.income_group.nunique() == 4, "Income groups missing or invalid"
    assert list(d.income_group.cat.categories) == INCOME and d.income_group.cat.ordered, "Incorrect income category order"
    assert d.survey_date.dropna().le(pd.Timestamp.now(tz="UTC")).all(), "Future survey date"
    assert d.health_spend_pct.notna().all(), "Unfilled health spending"
    # Preserve precisely the documented missing records and imputation provenance.
    assert d.isna().sum().to_dict() == {**{c: 0 for c in SCHEMA}, "population_m": 2, "gdp_pc_usd": 1, "life_expectancy": 2, "survey_date": 1}, "Unexpected missingness"
    assert int(d.health_spend_pct_imputed.sum()) == 29, "Incorrect imputation count"
    return True

assert validate(df)
# A same-sized copy isolates the duplicate-key failure from the row-count failure.
broken = df.copy()
broken.loc[1, KEY] = broken.loc[0, KEY].values
try:
    validate(broken)
except AssertionError as err:
    print("Expected validation failure:", err)
    assert "Duplicate" in str(err)
else:
    raise AssertionError("Validator failed to reject a duplicate")

saved = Path(f"data/processed/{GROUP}_clean.csv")
saved.parent.mkdir(parents=True, exist_ok=True)
# Use +00:00 (rather than +0000) and an explicit ISO date/time separator.
export = df.copy()
export["survey_date"] = df.survey_date.map(lambda x: x.isoformat() if pd.notna(x) else None)
export.to_csv(saved, index=False)
back_typed = pd.read_csv(saved, dtype={c: t for c,t in EXPECTED_DTYPES.items() if c not in ["income_group", "survey_date"]})
back_typed["income_group"] = back_typed.income_group.astype(pd.CategoricalDtype(INCOME, ordered=True))
back_typed["survey_date"] = pd.to_datetime(back_typed.survey_date, format="%Y-%m-%dT%H:%M:%S%z", utc=True)
assert validate(back_typed)
pd.testing.assert_frame_equal(df, back_typed)
assert hashlib.sha256(PATH.read_bytes()).hexdigest() == PUBLISHED_SHA256, "Raw source was modified"
print("Validated and round-trip checked:", saved, "| rows:", len(df))

Expected validation failure: Duplicate country-year key
Validated and round-trip checked: data/processed/6_clean.csv | rows: 125


In [21]:
# Self-check for checkpoint 9
saved = Path(f"data/processed/{GROUP}_clean.csv")
check(saved.exists(), "Save your clean data to data/processed/<GROUP>_clean.csv.")
if saved.exists():
    back = pd.read_csv(saved)
    print(back.dtypes)                      # reading a CSV: survey_date is text again, categories are plain text
    check(back["survey_date"].dropna().astype(str).str.endswith("+00:00").all(), "Write survey_date as a UTC datetime so the file records +00:00.")
    check(len(back) == 125 and not back.duplicated(["iso3", "year"]).any(), "The saved file should have 125 unique country-years.")

✅


country                      object
iso3                         object
year                          int64
population_m                float64
gdp_pc_usd                  float64
health_spend_pct            float64
life_expectancy             float64
income_group                 object
survey_date                  object
health_spend_pct_imputed       bool
dtype: object
✅
✅


### Decision log, checkpoint 9: validate and save

**Implementation:** AI-assisted draft prepared for Group 6. **Driver / navigators:** pending actual names and roles; no rotation is claimed.

| What we found (with a count) | What we decided | Justification (one line) |
|---|---|---|
| 125 rows, 25 countries × 5 years, 10 output columns including 1 imputation flag | Assert schema/order/dtypes, coverage, keys, ranges, canonical names, category order, UTC/no future dates and missingness | An unexpected change must stop execution rather than silently produce a plausible CSV. |
| 1 deliberately duplicated key in a same-sized test copy | Require validate to raise Duplicate country-year key | This isolates the key check from the row-count check; the clean table remains untouched. |
| 1 clean CSV written and read back; typed round-trip equals the in-memory table | Save data/processed/6_clean.csv with ISO 8601 UTC dates; explicitly restore dtypes on reload | CSV does not preserve categorical, nullable integer or datetime types. |
| Raw SHA-256 unchanged after cleaning | Leave tracked raw source untouched | The cleaning pipeline is reproducible from the original bytes. |

---
## Hand-in: commit, push and open a pull request (5 minutes)

1. **Runtime → Restart and run all.** Ensure that every cell in your notebook must run from the top. (Your reviewers will do exactly this.)
2. Upload this notebook into your group repository; add `data/processed/<GROUP>_clean.csv`. **Do not** commit the raw (original) file: `week4_country_panel_raw.csv`.
3. Share your repo with the reviewing group. Also share it with your tutor.

The tutor's rotation: A reviews B, B reviews C, and so on, round the room.

The tutor scores your saved file during the review round.

---
## Round 2: review another group's decisions (10 minutes)

Work in your own copy of this notebook. **Rotate roles again:** a new driver opens the other group's pull request; the navigators score.

1. **Re-run their notebook** in Colab: open it from their branch, *Restart and run all*. Does it run from top to bottom? Does it produce 125 rows? (4 points)
2. **Score their decision log** with the rubric below (this becomes their "Decision log" score, moderated by the tutor).
3. **Bug bounty:** for each problem in their **clean data** that they missed or got wrong, write one line of evidence (a row, a value, a count). Each one confirmed by the answer key earns **you** 3 points (up to 12).
4. **Comment on their pull request:** at least three specific, constructive comments on lines of code or log entries (4 points). Be specific and kind: "Checkpoint 6: JPN gdp_pc_usd is about 40 in every year: thousands of USD?" beats "units wrong".

### Rubric for their decision log (30 points)

| Item | Points |
|---|---|
| Each of checkpoints 1 to 9: a count, a decision and a one-line justification for every problem found (2 complete, 1 partial, 0 missing) | 9 × 2 = 18 |
| Checkpoint 8 argument: **evidence** used (share missing, pattern by group, mechanism) | 4 |
| Checkpoint 8 argument: the **other option** treated fairly | 4 |
| Checkpoint 8 argument: the **consequence** for an analysis stated | 4 |

In [22]:
# 🟡 Fill in your review, then run: it writes a file to commit to YOUR repository
REVIEWED_GROUP = None
log_scores = {"cp1": 0, "cp2": 0, "cp3": 0, "cp4": 0, "cp5": 0, "cp6": 0, "cp7": 0, "cp8": 0, "cp9": 0}   # 0, 1 or 2 each
argument = {"evidence": 0, "other_option": 0, "consequence": 0}                                         # 0 to 4 each
reran_from_top = None          # add True if the notebook run or False if it didn't
rerun_note = "Pending: no other group notebook has been supplied or reviewed."            # one line: what happened when you ran it. It it runs, you can just add "It run", otherwise "It did not run"
bounties = [                   # one line of evidence per problem they missed, e.g.
    # "cp6: JPN gdp_pc_usd about 40 in every year: still thousands of USD",
]
pr_comments_made = 0           # how many comments you left on their pull request

review = {"reviewer": GROUP, "reviewed": REVIEWED_GROUP, "log_scores": log_scores, "argument": argument,
          "log_total": sum(log_scores.values()) + sum(argument.values()), "reran_from_top": reran_from_top,
          "rerun_note": rerun_note, "bounties": bounties, "pr_comments_made": pr_comments_made}
if REVIEWED_GROUP is None:
    print("Peer review pending: provide the assigned group's notebook/PR before recording scores or comments.")
else:
    Path("reviews").mkdir(exist_ok=True)
    Path(f"reviews/{GROUP}_reviews_{REVIEWED_GROUP}.json").write_text(json.dumps(review, indent=2))
    print(json.dumps(review, indent=2))

Peer review pending: provide the assigned group's notebook/PR before recording scores or comments.


Commit `reviews/<GROUP>_reviews_<REVIEWED>.json` to **your** repository and push. The tutor collects every review file for the leaderboard.

---
## Round 3: the debate (one minute per group)

One speaker per group (someone who has not spoken yet today) argues your checkpoint 8 decision in **one minute**: the evidence, why not the other option, the consequence. The class votes on Mentimeter for the **best argument**, whichever side it is on. The winner takes the "Best argument" prize.

**Speaker:** pending actual group choice. **Proposed one-minute outline:**

- **Evidence:** 29/125 health values are missing; rates fall from 60% in low-income observations to 2.22% in high-income observations. Missingness depends on observed group membership; MAR is an assumption, not a finding.
- **Choice and alternative:** impute within income group with a flag to retain the variable; dropping it avoids manufactured values but loses the ability to study health expenditure.
- **Consequence:** medians compress variation and may bias relationships. Present imputed values as estimates and compare observed-only results; low-income conclusions are especially sensitive.

**Pending real-world tasks:** confirm actual driver/navigator roles, review the assigned other group when its link is available, and agree the final debate position. No peer scores, reviewer comments, commits, pushes or submission are claimed.

---
## Before you leave

- [ ] Every checkpoint log has a count, a decision and a one-line justification.
- [ ] Your notebook runs from the top; your clean CSV file and review file are pushed.
- [ ] You responded to your reviewers' comments on your pull request (before next week).